In [6]:
# ============================================================
# PROCESS 04 - INITIAL RISK SCREENING GATEWAY (COMPLETE)
# Purpose: Screen patients BEFORE ECG upload using lifestyle/clinical data
# Model: Random Forest | XAI: SHAP | API: Flask/FastAPI Ready
# Output: 0 (Low Risk - Optional ECG) or 1 (High Risk - Mandatory ECG)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

# Install SHAP
!pip install shap -q

print("✅ Initial Risk Screening Gateway - Complete System")
print("🎯 API Ready for Frontend Integration")
print("="*60)

# ============================================================
# 1. LOAD & PREPROCESS DATA (Pre-ECG Only)
# ============================================================

#from google.colab import drive
#drive.mount('/content/drive')
df = pd.read_csv(r"C:\Users\User\Desktop\DSGP Viva\model\CM2603-DSGP-Group-3\Dataset\Heart_health new.csv")

print(f"📊 Original dataset: {df.shape[0]} patients")

# Clean duplicates
df = df.drop_duplicates()

# Feature Engineering
df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure mmHg'].str.split('/', expand=True).astype(int)
df['Height_m'] = df['Height cm'] / 100
df['BMI'] = df['Weight kg'] / (df['Height_m'] ** 2)

# Add Lifestyle Factors
np.random.seed(42)
n_samples = len(df)

lifestyle_data = {
    'Alcohol': np.random.choice(['None', 'Light', 'Heavy'], n_samples, p=[0.5, 0.4, 0.1]),
    'Physical_Activity': np.random.choice(['Sedentary', 'Moderate', 'Active'], n_samples, p=[0.25, 0.55, 0.2]),
    'Family_History': np.random.choice([0, 1], n_samples, p=[0.75, 0.25]),
    'Diabetes': np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
    'Stress_Level': np.random.choice(['Low', 'Medium', 'High'], n_samples, p=[0.3, 0.5, 0.2]),
    'Sleep_Hours': np.clip(np.random.normal(7, 1.5, n_samples), 4, 10).round(1),
    'Years_Smoking': np.where(df['Smoker'] == 'Yes', np.random.randint(1, 35, n_samples), 0)
}

for col, vals in lifestyle_data.items():
    df[col] = vals

# Risk Flags
df['High_Cholesterol'] = (df['Cholesterol mg/dL'] > 200).astype(int)
df['High_Glucose'] = (df['Glucose mg/dL'] > 100).astype(int)
df['Hypertension'] = ((df['Systolic_BP'] >= 140) | (df['Diastolic_BP'] >= 90)).astype(int)
df['Obesity'] = (df['BMI'] >= 30).astype(int)

# Risk Score
def calc_risk_score(row):
    score = 0
    if row['Age'] > 50: score += 1
    if row['Smoker'] == 'Yes': score += 1
    if row['Diabetes'] == 1: score += 1
    if row['High_Cholesterol'] == 1: score += 1
    if row['Hypertension'] == 1: score += 1
    if row['Obesity'] == 1: score += 1
    if row['Exercise hours/week'] < 2: score += 1
    return score

df['Risk_Score'] = df.apply(calc_risk_score, axis=1)

# Encode Categoricals
encoders = {}
categorical_cols = ['Gender', 'Smoker', 'Alcohol', 'Physical_Activity', 'Stress_Level']

for col in categorical_cols:
    le = LabelEncoder()
    df[f'{col}_Encoded'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# Gateway Features
GATEWAY_FEATURES = [
    'Age', 'Gender_Encoded', 'BMI', 'Systolic_BP', 'Diastolic_BP',
    'Cholesterol mg/dL', 'Glucose mg/dL',
    'Smoker_Encoded', 'Diabetes', 'Alcohol_Encoded', 'Physical_Activity_Encoded',
    'Family_History', 'Stress_Level_Encoded', 'Sleep_Hours', 'Years_Smoking',
    'Exercise hours/week',
    'High_Cholesterol', 'High_Glucose', 'Hypertension', 'Obesity',
    'Risk_Score'
]

X = df[GATEWAY_FEATURES]
y = df['Heart Attack']

print(f"✅ Gateway features: {len(GATEWAY_FEATURES)}")

# ============================================================
# 2. TRAIN MODEL
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

gateway_model = RandomForestClassifier(
    n_estimators=100, max_depth=6, min_samples_split=10,
    min_samples_leaf=4, max_features='sqrt', class_weight='balanced', random_state=42
)

gateway_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = gateway_model.predict(X_test_scaled)
print(f"\n📊 Test Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"📊 Test F1-Score: {f1_score(y_test, y_pred):.4f}")

# SHAP
explainer = shap.TreeExplainer(gateway_model)

# ============================================================
# 3. INPUT HANDLING FUNCTIONS
# ============================================================

def calculate_derived_features(raw_input):
    """
    Calculate BMI, Risk Flags, and Risk Score from raw inputs
    """
    # Calculate BMI
    height_m = raw_input['height_cm'] / 100
    bmi = raw_input['weight_kg'] / (height_m ** 2)

    # Parse BP
    systolic, diastolic = map(int, raw_input['blood_pressure'].split('/'))

    # Risk Flags
    high_chol = 1 if raw_input['cholesterol'] > 200 else 0
    high_glucose = 1 if raw_input['glucose'] > 100 else 0
    hypertension = 1 if (systolic >= 140 or diastolic >= 90) else 0
    obesity = 1 if bmi >= 30 else 0

    # Risk Score
    risk_score = sum([
        raw_input['age'] > 50,
        raw_input['smoker'] == 'Yes',
        raw_input['diabetes'] == 'Yes',
        high_chol,
        hypertension,
        obesity,
        raw_input['exercise_hours'] < 2
    ])

    return {
        'BMI': round(bmi, 1),
        'Systolic_BP': systolic,
        'Diastolic_BP': diastolic,
        'High_Cholesterol': high_chol,
        'High_Glucose': high_glucose,
        'Hypertension': hypertension,
        'Obesity': obesity,
        'Risk_Score': risk_score
    }


def process_raw_input(raw_input):
    """
    Convert frontend form data to model-ready format
    raw_input: dict from HTML form / API request
    """
    # Calculate derived features
    derived = calculate_derived_features(raw_input)

    # Encode categoricals
    gender_encoded = encoders['Gender'].transform([raw_input['gender']])[0]
    smoker_encoded = encoders['Smoker'].transform([raw_input['smoker']])[0]
    alcohol_encoded = encoders['Alcohol'].transform([raw_input['alcohol']])[0]
    activity_encoded = encoders['Physical_Activity'].transform([raw_input['physical_activity']])[0]
    stress_encoded = encoders['Stress_Level'].transform([raw_input['stress_level']])[0]

    # Build feature vector
    patient = {
        'Age': raw_input['age'],
        'Gender_Encoded': gender_encoded,
        'BMI': derived['BMI'],
        'Systolic_BP': derived['Systolic_BP'],
        'Diastolic_BP': derived['Diastolic_BP'],
        'Cholesterol mg/dL': raw_input['cholesterol'],
        'Glucose mg/dL': raw_input['glucose'],
        'Smoker_Encoded': smoker_encoded,
        'Diabetes': 1 if raw_input['diabetes'] == 'Yes' else 0,
        'Alcohol_Encoded': alcohol_encoded,
        'Physical_Activity_Encoded': activity_encoded,
        'Family_History': 1 if raw_input['family_history'] == 'Yes' else 0,
        'Stress_Level_Encoded': stress_encoded,
        'Sleep_Hours': raw_input['sleep_hours'],
        'Years_Smoking': raw_input['years_smoking'] if raw_input['smoker'] == 'Yes' else 0,
        'Exercise hours/week': raw_input['exercise_hours'],
        'High_Cholesterol': derived['High_Cholesterol'],
        'High_Glucose': derived['High_Glucose'],
        'Hypertension': derived['Hypertension'],
        'Obesity': derived['Obesity'],
        'Risk_Score': derived['Risk_Score']
    }

    return patient


def screen_patient(patient_dict):
    """
    Main Gateway API - Returns screening result with explanation
    """
    # Convert to DataFrame
    input_df = pd.DataFrame([patient_dict])
    X_input = input_df[GATEWAY_FEATURES]
    X_input_scaled = scaler.transform(X_input)

    # Predict
    risk_prob = gateway_model.predict_proba(X_input_scaled)[0][1]
    risk_class = gateway_model.predict(X_input_scaled)[0]

    # SHAP explanation
    shap_vals = explainer.shap_values(X_input_scaled)

    if isinstance(shap_vals, list):
        class_1_impacts = shap_vals[1][0]
    else:
        class_1_impacts = shap_vals[0, :, 1]

    # Top 3 factors
    contributions = [(f, class_1_impacts[i], X_input.iloc[0][f])
                     for i, f in enumerate(GATEWAY_FEATURES)]
    contributions.sort(key=lambda x: abs(x[1]), reverse=True)

    top_factors = []
    for feature, impact, value in contributions[:3]:
        effect = "higher risk" if impact > 0 else "lower risk"
        # Decode categorical
        display_value = value
        if '_Encoded' in feature:
            orig = feature.replace('_Encoded', '')
            if orig in encoders:
                try:
                    display_value = encoders[orig].inverse_transform([int(value)])[0]
                except:
                    pass
        top_factors.append({
            'factor': feature.replace('_Encoded', '').replace('_', ' '),
            'impact': round(impact, 3),
            'value': str(display_value),
            'effect': effect
        })

    # Decision
    if risk_class == 1:
        return {
            'success': True,
            'risk_status': 'HIGH_RISK',
            'risk_probability': round(float(risk_prob), 4),
            'decision': 'MANDATORY_ECG',
            'message': "You show signs of elevated CAD risk based on your profile. Please upload your ECG image for detailed analysis.",
            'explanation': top_factors,
            'next_step': 'UPLOAD_ECG',
            'requires_ecg': True,
            'color': 'red'
        }
    else:
        return {
            'success': True,
            'risk_status': 'LOW_RISK',
            'risk_probability': round(float(risk_prob), 4),
            'decision': 'OPTIONAL_ECG',
            'message': "You do not currently show significant CAD risk. You may upload an ECG for additional verification if you wish.",
            'explanation': top_factors,
            'next_step': 'OPTIONAL_UPLOAD',
            'requires_ecg': False,
            'color': 'green'
        }


def interactive_input():
    """
    Console-based input for testing
    """
    print("\n" + "="*60)
    print("🏥 CAD RISK SCREENING - PATIENT REGISTRATION")
    print("="*60)

    raw = {}
    print("\n📋 Basic Information")
    raw['age'] = int(input("Age (years): "))
    raw['gender'] = input("Gender (Male/Female): ").strip()

    print("\n📋 Body Measurements")
    raw['height_cm'] = float(input("Height (cm): "))
    raw['weight_kg'] = float(input("Weight (kg): "))

    print("\n📋 Clinical Measurements")
    raw['blood_pressure'] = input("Blood Pressure (e.g., 120/80): ").strip()
    raw['cholesterol'] = int(input("Cholesterol (mg/dL): "))
    raw['glucose'] = int(input("Glucose (mg/dL): "))

    print("\n📋 Lifestyle Factors")
    raw['smoker'] = input("Do you smoke? (Yes/No): ").strip()
    raw['years_smoking'] = int(input("Years smoking (0 if non-smoker): ")) if raw['smoker'] == 'Yes' else 0
    raw['diabetes'] = input("Diabetes? (Yes/No): ").strip()
    raw['alcohol'] = input("Alcohol consumption (None/Light/Heavy): ").strip()
    raw['physical_activity'] = input("Physical Activity (Sedentary/Moderate/Active): ").strip()
    raw['family_history'] = input("Family history of CAD? (Yes/No): ").strip()
    raw['stress_level'] = input("Stress Level (Low/Medium/High): ").strip()
    raw['sleep_hours'] = float(input("Sleep hours per night: "))
    raw['exercise_hours'] = float(input("Exercise hours per week: "))

    return raw


def print_report(result):
    """Display formatted report"""
    print("\n" + "="*60)
    print("🩺 SCREENING REPORT")
    print("="*60)
    print(f"Status: {result['risk_status']}")
    print(f"Probability: {result['risk_probability']:.1%}")
    print(f"Decision: {result['decision']}")
    print("-"*60)
    print(f"Message: {result['message']}")
    print("-"*60)
    print("Key Factors:")
    for f in result['explanation']:
        emoji = "🔴" if f['effect'] == 'higher risk' else "🟢"
        print(f"  {emoji} {f['factor']}: {f['value']} ({f['impact']})")
    print("="*60)

# ============================================================
# 4. DEMONSTRATION
# ============================================================

print("\n" + "="*60)
print("INTERACTIVE DEMONSTRATION")
print("="*60)

# Get input from user
raw_patient = interactive_input()

# Process and screen
processed = process_raw_input(raw_patient)
result = screen_patient(processed)
print_report(result)

# ============================================================
# 5. SAVE MODEL ARTIFACTS
# ============================================================

print("\n" + "="*60)
print("SAVING MODEL ARTIFACTS")
print("="*60)

pickle.dump(gateway_model, open('gateway_model.pkl', 'wb'))
pickle.dump(scaler, open('gateway_scaler.pkl', 'wb'))
pickle.dump(encoders, open('gateway_encoders.pkl', 'wb'))
pickle.dump(GATEWAY_FEATURES, open('gateway_features.pkl', 'wb'))

print("✅ Saved: gateway_model.pkl, gateway_scaler.pkl, gateway_encoders.pkl, gateway_features.pkl")



✅ Initial Risk Screening Gateway - Complete System
🎯 API Ready for Frontend Integration
📊 Original dataset: 724 patients
✅ Gateway features: 21

📊 Test Accuracy: 98.88%
📊 Test F1-Score: 0.9906

INTERACTIVE DEMONSTRATION

🏥 CAD RISK SCREENING - PATIENT REGISTRATION

📋 Basic Information


'pip' is not recognized as an internal or external command,
operable program or batch file.



📋 Body Measurements

📋 Clinical Measurements

📋 Lifestyle Factors

🩺 SCREENING REPORT
Status: HIGH_RISK
Probability: 58.2%
Decision: MANDATORY_ECG
------------------------------------------------------------
Message: You show signs of elevated CAD risk based on your profile. Please upload your ECG image for detailed analysis.
------------------------------------------------------------
Key Factors:
  🟢 High Cholesterol: 0.0 (-0.137)
  🟢 Cholesterol mg/dL: 200.0 (-0.117)
  🔴 Glucose mg/dL: 220.0 (0.075)

SAVING MODEL ARTIFACTS
✅ Saved: gateway_model.pkl, gateway_scaler.pkl, gateway_encoders.pkl, gateway_features.pkl
